In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 5 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Function 5 has a very strong corner solution:
#
#     [1, 1, 1, 1]
#
# Recent ~0.01 one-coordinate moves away from the corner have
# all reduced the objective.
#
# Therefore Week 12 focuses on finding the LEAST damaging
# admissible move just outside the duplicate radius.
#
# Strategy:
# - manually standardise Y
# - refit ARD Matern GP
# - check Week 11 calibration
# - search a thin shell around the incumbent
# - compare with four one-coordinate rays
# - retain wider/global candidates only as diagnostics
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 12 data
# ------------------------------------------------------------

X = np.load(
    "function5/initial_inputs.npy"
)

Y = np.load(
    "function5/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration
# ------------------------------------------------------------
#
# Week 11 selected:
# [1.000000, 0.989999, 1.000000, 1.000000]
#
# Previous GP prediction:
# mean = 8407.005792278625
# std  = 132.82485810592786
#
# Actual:
# 8472.613847009368
# ------------------------------------------------------------

week11_pred_mean = (
    8407.005792278625
)

week11_pred_std = (
    132.82485810592786
)

week11_actual = (
    8472.613847009368
)

week11_error = (
    week11_actual
    - week11_pred_mean
)

week11_z_error = (
    week11_error
    / week11_pred_std
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week11_pred_mean
)

print(
    "Predicted std :",
    week11_pred_std
)

print(
    "Actual        :",
    week11_actual
)

print(
    "\nPrediction error:",
    week11_error
)

print(
    "\nError / predicted std:",
    week11_z_error
)


# ------------------------------------------------------------
# 3. Show observations nearest to the incumbent
# ------------------------------------------------------------

dist_from_best = np.linalg.norm(
    X - best_x,
    axis=1
)

order = np.argsort(
    dist_from_best
)

print("\n================================")
print("NEAREST OBSERVED POINTS")
print("================================")

for idx in order[:8]:

    print(
        "\nx =",
        X[idx]
    )

    print(
        "distance =",
        dist_from_best[idx]
    )

    print(
        "y =",
        Y[idx]
    )


# ------------------------------------------------------------
# 4. Manual Y standardisation
# ------------------------------------------------------------

y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (
    Y - y_mean
) / y_std

best_y_scaled = (
    best_y - y_mean
) / y_std

print("\n================================")
print("Y STANDARDISATION")
print("================================")

print("Y mean:", y_mean)
print("Y std :", y_std)

print(
    "\nBest scaled Y:",
    best_y_scaled
)


# ------------------------------------------------------------
# 5. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 6. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 7. Candidate generator
# ------------------------------------------------------------

rng = np.random.default_rng(42)

tree = cKDTree(X)


# ------------------------------------------------------------
# 8. Thin shell around [1,1,1,1]
# ------------------------------------------------------------
#
# Instead of only perturbing one coordinate, sample directions
# inside the feasible lower-facing quadrant.
#
# Radius starts just outside 0.01.
# ------------------------------------------------------------

n_shell = 300000

directions = np.abs(
    rng.normal(
        size=(n_shell, 4)
    )
)

directions = (
    directions
    / np.linalg.norm(
        directions,
        axis=1,
        keepdims=True
    )
)

radii = rng.uniform(
    0.010001,
    0.04,
    size=n_shell
)

shell_candidates = (
    best_x
    - directions
    * radii[:, None]
)

shell_candidates = np.clip(
    shell_candidates,
    0,
    1
)


# ------------------------------------------------------------
# 9. Explicit one-coordinate rays
# ------------------------------------------------------------

ray_values = np.arange(
    0.94,
    0.990000,
    0.000001
)

ray_x1 = np.ones(
    (len(ray_values), 4)
)
ray_x1[:, 0] = ray_values

ray_x2 = np.ones(
    (len(ray_values), 4)
)
ray_x2[:, 1] = ray_values

ray_x3 = np.ones(
    (len(ray_values), 4)
)
ray_x3[:, 2] = ray_values

ray_x4 = np.ones(
    (len(ray_values), 4)
)
ray_x4[:, 3] = ray_values


# ------------------------------------------------------------
# 10. Wider diagnostic pool
# ------------------------------------------------------------

wide_candidates = (
    best_x
    + rng.normal(
        0,
        0.08,
        size=(100000, 4)
    )
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)


# ------------------------------------------------------------
# 11. Global diagnostic pool
# ------------------------------------------------------------

global_candidates = rng.uniform(
    0,
    1,
    size=(70000, 4)
)


# ------------------------------------------------------------
# 12. Combine + round to PORTAL precision
# ------------------------------------------------------------
#
# Important:
# filtering is done AFTER rounding to six decimals so that
# duplicate distances match the values we could actually submit.
# ------------------------------------------------------------

candidates = np.vstack([
    shell_candidates,
    ray_x1,
    ray_x2,
    ray_x3,
    ray_x4,
    wide_candidates,
    global_candidates
])

candidates = np.round(
    candidates,
    6
)


# ------------------------------------------------------------
# 13. Near-duplicate filtering
# ------------------------------------------------------------

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\n================================")
print("CANDIDATES")
print("================================")

print(
    "Candidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 14. Predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled
    * y_std
    + y_mean
)

sigma_raw = (
    sigma_scaled
    * y_std
)


# ------------------------------------------------------------
# 15. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "raw mean =",
    mu_raw[ei_idx]
)

print(
    "raw std =",
    sigma_raw[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from best =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 16. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu_scaled
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "raw mean =",
    mu_raw[mean_idx]
)

print(
    "raw std =",
    sigma_raw[mean_idx]
)

print(
    "distance from best =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 17. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "raw mean =",
        mu_raw[idx]
    )

    print(
        "raw std =",
        sigma_raw[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 18. Best point on each ray
# ------------------------------------------------------------

print("\n================================")
print("BEST PREDICTED MEAN ON EACH RAY")
print("================================")

ray_names = [
    "x1",
    "x2",
    "x3",
    "x4"
]

ray_arrays = [
    ray_x1,
    ray_x2,
    ray_x3,
    ray_x4
]

for name, ray in zip(
    ray_names,
    ray_arrays
):

    ray = np.round(
        ray,
        6
    )

    ray_distance, _ = tree.query(
        ray,
        k=1
    )

    ray_valid = ray[
        ray_distance > 0.01
    ]

    if len(ray_valid) == 0:

        print(
            f"\n{name}: no admissible points"
        )

        continue

    ray_mu_scaled, ray_sigma_scaled = gp.predict(
        ray_valid,
        return_std=True
    )

    ray_mu_raw = (
        ray_mu_scaled
        * y_std
        + y_mean
    )

    ray_sigma_raw = (
        ray_sigma_scaled
        * y_std
    )

    idx = np.argmax(
        ray_mu_scaled
    )

    print(
        f"\n{name} ray:"
    )

    print(
        "candidate =",
        ray_valid[idx]
    )

    print(
        "raw mean =",
        ray_mu_raw[idx]
    )

    print(
        "raw std =",
        ray_sigma_raw[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            ray_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 19. Best candidate from shell only
# ------------------------------------------------------------

shell_candidates = np.round(
    shell_candidates,
    6
)

shell_distance, _ = tree.query(
    shell_candidates,
    k=1
)

shell_valid = shell_candidates[
    shell_distance > 0.01
]

shell_mu_scaled, shell_sigma_scaled = gp.predict(
    shell_valid,
    return_std=True
)

shell_mu_raw = (
    shell_mu_scaled
    * y_std
    + y_mean
)

shell_sigma_raw = (
    shell_sigma_scaled
    * y_std
)

shell_idx = np.argmax(
    shell_mu_scaled
)

print("\n================================")
print("BEST THIN-SHELL CANDIDATE")
print("================================")

print(
    "candidate =",
    shell_valid[shell_idx]
)

print(
    "raw mean =",
    shell_mu_raw[shell_idx]
)

print(
    "raw std =",
    shell_sigma_raw[shell_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        shell_valid[shell_idx]
        - best_x
    )
)

DATA
X shape: (31, 4)
Y shape: (31,)

Current best:
[1. 1. 1. 1.] -> 8662.4825

Y range:
min = 0.1129397953712203
max = 8662.4825
std = 3273.10408487822

WEEK 11 CALIBRATION CHECK
Predicted mean: 8407.005792278625
Predicted std : 132.82485810592786
Actual        : 8472.613847009368

Prediction error: 65.60805473074288

Error / predicted std: 0.4939440980122895

NEAREST OBSERVED POINTS

x = [1. 1. 1. 1.]
distance = 0.0
y = 8662.4825

x = [1.       0.989999 1.       1.      ]
distance = 0.010001000000000038
y = 8472.613847009368

x = [1.       1.       1.       0.989992]
distance = 0.010008000000000017
y = 8472.483647641197

x = [1.       1.       0.989971 1.      ]
distance = 0.010028999999999955
y = 8472.09307186791

x = [1.       1.       1.       0.979991]
distance = 0.020009000000000055
y = 8290.218689357127

x = [0.949995 1.       1.       1.      ]
distance = 0.050004999999999966
y = 7786.29242354451

x = [0.729149 1.       1.       1.      ]
distance = 0.27085099999999995
y = 551

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.68**2 * Matern(length_scale=[1.25, 1.32, 1.25, 2], nu=2.5) + WhiteKernel(noise_level=0.000913)

ARD lengthscales:
[1.24812613 1.31647148 1.24868324 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.27997862 0.26544337 0.2798537  0.17472431]

CANDIDATES
Candidates after portal rounding + duplicate filtering:
549140

PRIMARY EI
candidate = [1.       1.       0.659686 1.      ]
raw mean = 6182.932226423147
raw std = 986.5287963661535
EI = 0.0005794295234454773
distance from best = 0.340314

HIGHEST PREDICTED MEAN
candidate = [1.       1.       0.989452 0.988987]
raw mean = 8383.956479054961
raw std = 112.57721175430805
distance from best = 0.01524947451553663

UCB DIAGNOSTICS

beta=0.05
candidate = [1.       1.       0.989452 0.988987]
raw mean = 8383.956479054961
raw std = 112.57721175430805
distance = 0.01524947451553663

beta=0.1
candidate = [1.       1.       0.989452 0.988987]
raw mean = 8383.956479054961
raw std = 112.57721175430805
distance = 0.0

In [2]:
# ============================================================
# FINAL FUNCTION 5 - WEEK 12 SELECTION
# ============================================================
#
# The unrestricted posterior mean / UCB candidate makes small
# simultaneous moves in x3 and x4.
#
# However:
# - x3, x4 and x2 have already been tested near the corner
#   and each reduced performance.
# - The predicted advantage of the unrestricted candidate over
#   the x1 ray is tiny relative to posterior uncertainty.
# - x1 is the remaining near-corner direction that has not yet
#   been tested with a ~0.01 perturbation.
#
# Therefore choose a controlled, portal-safe x1-only move.
# ============================================================

week12_candidate = np.array([
    0.989999,
    1.000000,
    1.000000,
    1.000000
])

print("================================")
print("FUNCTION 5 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction on scaled output
final_mu_scaled, final_sigma_scaled = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

# Convert back to raw objective scale
final_mu_raw = (
    final_mu_scaled[0] * y_std
    + y_mean
)

final_sigma_raw = (
    final_sigma_scaled[0] * y_std
)

print("\nPredicted raw mean:")
print(final_mu_raw)

print("\nPredicted raw std:")
print(final_sigma_raw)

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

# Check nearest historical observation after portal rounding
nearest_dist, _ = tree.query(
    week12_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 5 - WEEK 12 FINAL

Candidate:
[0.989999 1.       1.       1.      ]

Predicted raw mean:
8383.491485915245

Predicted raw std:
107.49164619316507

Distance from current best:
0.010001000000000038

Nearest observed-point distance:
0.010001000000000038

Portal format:
0.989999-1.000000-1.000000-1.000000
